In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
#data download and preprocessing

from datasets import load_dataset
from datasets import concatenate_datasets
from transformers import AutoTokenizer
from pympler import asizeof
from torch.utils.data import DataLoader
from collections import Counter

# Load datasets
mnli_dataset = load_dataset("glue", "mnli")
snli_dataset = load_dataset("snli")

# Remove 'idx' column from MNLI
mnli_dataset = {
    split: ds.remove_columns("idx") 
    for split, ds in mnli_dataset.items()
}

# Function to filter out invalid labels
def filter_valid(example):
    return example["label"] != -1

# Filter all SNLI splits - WITH MULTIPROCESSING
snli_dataset = {
    split: ds.filter(filter_valid, num_proc=4)  # Add num_proc
    for split, ds in snli_dataset.items()
}

# Filter all MNLI splits except test split - WITH MULTIPROCESSING
mnli_dataset = {
    split: ds.filter(filter_valid, num_proc=4) if "test" not in split else ds  # Add num_proc
    for split, ds in mnli_dataset.items()
}


train_dataset = concatenate_datasets([
    snli_dataset['train'], 
    mnli_dataset['train']
]).shuffle(seed=42)


validation_dataset = concatenate_datasets([
    snli_dataset['validation'], 
    mnli_dataset['validation_matched']
]).shuffle(seed=42)


validation_mismatched_dataset = mnli_dataset['validation_mismatched']


# Check the dataset structure
print(mnli_dataset)
print(snli_dataset)

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# tokenize_function with dynamic padding
def tokenize_function(example):
    return tokenizer(
        example["premise"],
        example["hypothesis"],
        padding=False,
        truncation=True,
        max_length=128
    )

# Tokenize each dataset separately - WITH MULTIPROCESSING
tokenized_train = train_dataset.map(tokenize_function, batched=True, num_proc=4)
tokenized_validation = validation_dataset.map(tokenize_function, batched=True, num_proc=4)
tokenized_validation_mismatched = validation_mismatched_dataset.map(tokenize_function, batched=True, num_proc=4)

# Tokenize test sets - WITH MULTIPROCESSING
tokenized_test_mnli_matched = mnli_dataset["test_matched"].map(tokenize_function, batched=True, num_proc=4)
tokenized_test_mnli_mismatched = mnli_dataset["test_mismatched"].map(tokenize_function, batched=True, num_proc=4)
tokenized_test_snli = snli_dataset["test"].map(tokenize_function, batched=True, num_proc=4)


# Change dataset format to use "labels" instead of "label"
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_validation = tokenized_validation.rename_column("label", "labels")
tokenized_validation_mismatched = tokenized_validation_mismatched.rename_column("label", "labels")

tokenized_test_mnli_matched = tokenized_test_mnli_matched.rename_column("label", "labels")
tokenized_test_mnli_mismatched = tokenized_test_mnli_mismatched.rename_column("label", "labels")
tokenized_test_snli = tokenized_test_snli.rename_column("label", "labels")

# Set format for PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_validation.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_validation_mismatched.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])

tokenized_test_mnli_matched.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_test_mnli_mismatched.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_test_snli.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])


print(f"Train dataset size: {len(tokenized_train)}")
print(f"Validation dataset size: {len(tokenized_validation)}")
print(f"Validation mismatched size: {len(tokenized_validation_mismatched)}")

# attention_mask check 
print(tokenized_train[0]["attention_mask"])

# Check the train dataset dtype and size
size_in_bytes = asizeof.asizeof(tokenized_train)
size_in_mb = size_in_bytes / (1024 * 1024)
print(f"Total size of tokenized_train: {size_in_mb:.2f} MB")

# Check label distribution
train_labels = tokenized_train["labels"].tolist()
print(f"Label distribution in train: {Counter(train_labels)}")

print(tokenized_train[0]["input_ids"].shape, tokenized_train[0]["input_ids"].dtype)
print(tokenized_train[0]["token_type_ids"].shape, tokenized_train[0]["token_type_ids"].dtype)
print(tokenized_train[0]["attention_mask"].shape, tokenized_train[0]["attention_mask"].dtype)
print(tokenized_train[0]["labels"].shape, tokenized_train[0]["labels"].dtype)

# Check data types of each column
sample = tokenized_train[0]
for key, value in sample.items():
    print(f"Column: {key}, Data Type: {value.dtype}")

README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

mnli/test_matched-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

...

Map (num_proc=4):   0%|          | 0/9832 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9796 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9847 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9824 [00:00<?, ? examples/s]

Train dataset size: 942069
Validation dataset size: 19657
Validation mismatched size: 9832
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
Total size of tokenized_train: 1259.59 MB
Label distribution in train: Counter({0: 314315, 2: 314090, 1: 313664})
torch.Size([13]) torch.int64
torch.Size([13]) torch.int64
torch.Size([13]) torch.int64
torch.Size([]) torch.int64
Column: labels, Data Type: torch.int64
Column: input_ids, Data Type: torch.int64
Column: token_type_ids, Data Type: torch.int64
Column: attention_mask, Data Type: torch.int64


In [2]:
!pip install datasets transformers evaluate fvcore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.1 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 1.4 MB/s eta 0:00:00a 0:00:01
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=e835eed32dabe14e2d774d85feae929003271e279f16edeae52aa0b53b4741bb
  Stored in directory: /root/.cache/pip/wheels/65/71/95/3b8fde5c65c6e4a806e0867c1651dcc71a1cb2f3430e8f355f
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=b5cbf84c31a574f68377249dd2302e007c380ccd4b93f6cbd36f547ced034096
  Stored in directory: /root/.cache/pip/wheels/ba/5e/16/6117f8fe7e9c0c161a795e10d94645ebcf301ccbd01f66d8ec
Successfully built fvcore iopath
  Attempting uninstal

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import time
import math
from datetime import datetime, timezone
import inspect
from datasets import load_dataset, concatenate_datasets
from pympler import asizeof
from collections import Counter

# Custom Model class to integrate LoRA with your training loop
class BertLoRAForNLI(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_labels=3):
        super().__init__()
        
        # Load base model
        self.base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name, 
            num_labels=num_labels
        )
        
        # LoRA Configuration
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            inference_mode=False,
            r=8,  # Rank
            lora_alpha=16,  # Scaling parameter
            lora_dropout=0.1,
            target_modules=["query", "value"],  # Apply LoRA to attention layers
        )
        
        # Apply LoRA to the model
        self.model = get_peft_model(self.base_model, lora_config)
        
        # Print trainable parameters
        self.model.print_trainable_parameters()
    
    def forward(self, input_ids, token_type_ids, labels, attention_mask=None):
        """Forward pass compatible with your training loop"""
        outputs = self.model(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs.logits, outputs.loss
    
    def configure_optimizers(self, weight_decay, learning_rate, device, verbose=False):
        # Step 1: Collect trainable parameters (excluding frozen ones)
        param_dict = {pn: p for pn, p in self.named_parameters()}
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
    
        # Step 2: Separate weight decay (2D+ tensors) and no decay (biases, LayerNorm)
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]  # Weight matrices
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]  # Biases, LayerNorm
    
        # Step 3: Define optimizer parameter groups
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},  # Apply weight decay
            {'params': nodecay_params, 'weight_decay': 0.0}  # No weight decay
        ]

        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and 'cuda' in device
        
        # Debugging - Print parameter stats (optional)
        if verbose:
            num_decay_params = sum(p.numel() for p in decay_params)
            num_nodecay_params = sum(p.numel() for p in nodecay_params)
            print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params/1e6:.2f}M parameters")
            print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params/1e6:.2f}M parameters")
            print(f"Using fused AdamW: {use_fused}")

        # Create AdamW optimizer
        optimizer = torch.optim.AdamW(
            optim_groups, 
            lr=learning_rate, 
            betas=(0.9, 0.999),  # Momentum values
            eps=1e-8,  # Small epsilon for numerical stability
            fused=use_fused  # Enable fused version if available
        )
    
        return optimizer

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"

# Set random seeds for reproducibility
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

# Initialize the model
model = BertLoRAForNLI(model_name="bert-base-uncased", num_labels=3)
model = model.to(device)


2025-10-06 15:50:28.165573: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759765828.352048      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759765828.406234      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


In [9]:
import torch
import time
import numpy as np
from fvcore.nn import FlopCountAnalysis
from transformers import AutoTokenizer

def benchmark_model(model, tokenizer, text1, text2, seq_len=128, use_mixed_precision=True):
    """
    Benchmarks a BERT-based LoRA model (e.g., BertLoRAForNLI) using real tokenized inputs with a fixed sequence length.
    
    Args:
        model: LoRA-adapted BERT model (e.g., BertLoRAForNLI).
        tokenizer: Preloaded tokenizer (e.g., BertTokenizer).
        text1 (str): First input sentence (e.g., premise for NLI).
        text2 (str): Second input sentence (e.g., hypothesis for NLI).
        seq_len (int): Sequence length for tokenization (default: 128).
        use_mixed_precision (bool): Whether to use mixed precision (bf16/fp16) if on CUDA.
    
    Returns:
        List of throughput results for different batch sizes.
    """
    device = next(model.parameters()).device
    
    # Setup mixed precision
    if use_mixed_precision and device.type == "cuda":
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        print(f"Using mixed precision: {dtype}")
    else:
        dtype = torch.float32
        print("Using full precision: float32")
    
    # Prepare real tokenized inputs
    inputs = tokenizer(
        text1,
        text2,
        max_length=seq_len,  # Fixed to 128
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = inputs['input_ids'].to(device)  # Shape: [1, seq_len]
    token_type_ids = inputs['token_type_ids'].to(device)  # Shape: [1, seq_len]
    attention_mask = inputs['attention_mask'].to(device)  # Shape: [1, seq_len]
    labels = torch.tensor([0], device=device)  # Example label (0 for contradiction)
    
    model.eval()
    
    print("="*70)
    print("MODEL STATISTICS")
    print("="*70)
    
    # Parameter Count
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
    
    # FLOPs Calculation
    print("\n" + "="*70)
    print("FLOPS ANALYSIS")
    print("="*70)
    
    with torch.no_grad():
        try:
            # For LoRA model, pass inputs matching the forward method
            flops = FlopCountAnalysis(model, (input_ids, token_type_ids, labels, attention_mask))
            total_flops = flops.total()
        except Exception as e:
            print(f"FLOPs analysis with fvcore failed: {e}. Using approximate BERT FLOPs with LoRA adjustment.")
            # Approximate FLOPs for BERT-base with LoRA
            S, H, A, F, L = 128, 768, 12, 3072, 12  # seq_len, hidden_size, num_heads, ffn_size, num_layers
            base_flops = L * (4 * S * H**2 + 2 * S**2 * H + 4 * S * H * F)  # Attention + FFN
            # LoRA adds adapters to query and value: 2 * r * H per head per layer
            lora_params_per_layer = 2 * 8 * H * A  # r=8, query+value
            lora_flops = L * lora_params_per_layer * S  # Approximate FLOPs for LoRA adapters
            total_flops = base_flops + lora_flops
            print("Using approximate FLOPs calculation for LoRA model.")
        
        print(f"Total FLOPs per forward pass: {total_flops/1e9:.2f} GFLOPs")
        print(f"FLOPs per sample: {total_flops/1e9:.2f} GFLOPs")
        print(f"Total FLOPs for sequence: {total_flops/1e12:.4f} TFLOPs")
    
    # Memory Usage
    print("\n" + "="*70)
    print("MEMORY USAGE")
    print("="*70)
    
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        if use_mixed_precision and device.type == "cuda":
            with torch.autocast(device_type='cuda', dtype=dtype):
                _ = model(input_ids, token_type_ids, labels, attention_mask)
        else:
            _ = model(input_ids, token_type_ids, labels, attention_mask)
    
    if device.type == "cuda":
        memory_allocated = torch.cuda.max_memory_allocated() / 1e6
        print(f"Peak GPU memory (batch_size=1): {memory_allocated:.2f} MB")
    else:
        print("Memory usage not measured (running on CPU).")
    
    # Latency Measurement
    print("\n" + "="*70)
    print("LATENCY MEASUREMENT (batch_size=1)")
    print("="*70)
    
    # Warmup
    print("Warming up...")
    with torch.no_grad():
        for _ in range(20):
            if use_mixed_precision and device.type == "cuda":
                with torch.autocast(device_type='cuda', dtype=dtype):
                    _ = model(input_ids, token_type_ids, labels, attention_mask)
            else:
                _ = model(input_ids, token_type_ids, labels, attention_mask)
    if device.type == "cuda":
        torch.cuda.synchronize()
    
    # Actual measurement
    print("Measuring latency...")
    latencies = []
    num_iterations = 100
    
    with torch.no_grad():
        for _ in range(num_iterations):
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            
            if use_mixed_precision and device.type == "cuda":
                with torch.autocast(device_type='cuda', dtype=dtype):
                    _ = model(input_ids, token_type_ids, labels, attention_mask)
            else:
                _ = model(input_ids, token_type_ids, labels, attention_mask)
            
            if device.type == "cuda":
                torch.cuda.synchronize()
            latencies.append((time.perf_counter() - start) * 1000)  # ms
    
    latencies = np.array(latencies)
    mean_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    print(f"Latency: {mean_latency:.2f} ± {std_latency:.2f} ms")
    
    # Throughput Measurement
    print("\n" + "="*70)
    print("THROUGHPUT MEASUREMENT")
    print("="*70)
    
    batch_sizes = [1, 8, 16, 32]
    throughput_results = []
    
    for batch_size in batch_sizes:
        try:
            # Repeat tokenized inputs for larger batch sizes
            input_ids_batch = input_ids.repeat(batch_size, 1)
            token_type_ids_batch = token_type_ids.repeat(batch_size, 1)
            attention_mask_batch = attention_mask.repeat(batch_size, 1)
            labels_batch = labels.repeat(batch_size)
            
            # Warmup
            with torch.no_grad():
                for _ in range(10):
                    if use_mixed_precision and device.type == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, token_type_ids_batch, labels_batch, attention_mask_batch)
                    else:
                        _ = model(input_ids_batch, token_type_ids_batch, labels_batch, attention_mask_batch)
            if device.type == "cuda":
                torch.cuda.synchronize()
            
            # Measure
            num_iterations = 50
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            
            with torch.no_grad():
                for _ in range(num_iterations):
                    if use_mixed_precision and device.type == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, token_type_ids_batch, labels_batch, attention_mask_batch)
                    else:
                        _ = model(input_ids_batch, token_type_ids_batch, labels_batch, attention_mask_batch)
            
            if device.type == "cuda":
                torch.cuda.synchronize()
            elapsed_time = time.perf_counter() - start
            
            samples_per_sec = (batch_size * num_iterations) / elapsed_time
            tokens_per_sec = samples_per_sec * seq_len  # Use fixed seq_len=128
            
            # Memory check
            if device.type == "cuda":
                torch.cuda.reset_peak_memory_stats()
                with torch.no_grad():
                    if use_mixed_precision and device.type == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, token_type_ids_batch, labels_batch, attention_mask_batch)
                    else:
                        _ = model(input_ids_batch, token_type_ids_batch, labels_batch, attention_mask_batch)
                memory_used = torch.cuda.max_memory_allocated() / 1e9
            else:
                memory_used = None
            
            throughput_results.append({
                'batch_size': batch_size,
                'samples_per_sec': samples_per_sec,
                'tokens_per_sec': tokens_per_sec,
                'memory_gb': memory_used
            })
            
            memory_str = f"{memory_used:.2f} GB" if memory_used else "N/A (CPU)"
            print(f"Batch size {batch_size:3d}: {samples_per_sec:7.2f} samples/sec | "
                  f"{tokens_per_sec:10.2f} tokens/sec | Memory: {memory_str}")
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"Batch size {batch_size:3d}: OOM (Out of Memory)")
                if device.type == "cuda":
                    torch.cuda.empty_cache()
                break
            else:
                raise e
    
    # Performance Summary
    print("\n" + "="*70)
    print("PERFORMANCE SUMMARY")
    print("="*70)
    
    if throughput_results:
        max_throughput = max(throughput_results, key=lambda x: x['tokens_per_sec'])
        print(f"Peak throughput: {max_throughput['tokens_per_sec']:.2f} tokens/sec "
              f"(batch_size={max_throughput['batch_size']})")
        print(f"Single sample latency: {mean_latency:.2f} ± {std_latency:.2f} ms")
        print(f"FLOPs per forward pass: {total_flops/1e9:.2f} GFLOPs")
        
        # Theoretical achieved FLOPS
        mean_latency_sec = mean_latency / 1000
        theoretical_flops = total_flops / mean_latency_sec / 1e12
        print(f"Achieved compute: {theoretical_flops:.2f} TFLOPS")
    
    return throughput_results



# Define input texts
text1 = "The man is walking down the street."
text2 = "A person is outside."

results = benchmark_model(model, tokenizer, text1, text2, seq_len=128, use_mixed_precision=True)

Using mixed precision: torch.bfloat16
MODEL STATISTICS
Total parameters: 109,781,766 (109.78M)
Trainable parameters: 297,219 (0.30M)

FLOPS ANALYSIS
Total FLOPs per forward pass: 10.92 GFLOPs
FLOPs per sample: 10.92 GFLOPs
Total FLOPs for sequence: 0.0109 TFLOPs

MEMORY USAGE
Peak GPU memory (batch_size=1): 458.58 MB

LATENCY MEASUREMENT (batch_size=1)
Warming up...
Measuring latency...
Latency: 23.88 ± 1.07 ms

THROUGHPUT MEASUREMENT
Batch size   1:   42.86 samples/sec |    5486.36 tokens/sec | Memory: 0.46 GB
Batch size   8:  168.07 samples/sec |   21513.09 tokens/sec | Memory: 0.49 GB
Batch size  16:  187.58 samples/sec |   24010.68 tokens/sec | Memory: 0.53 GB
Batch size  32:  199.27 samples/sec |   25506.30 tokens/sec | Memory: 0.61 GB

PERFORMANCE SUMMARY
Peak throughput: 25506.30 tokens/sec (batch_size=32)
Single sample latency: 23.88 ± 1.07 ms
FLOPs per forward pass: 10.92 GFLOPs
Achieved compute: 0.46 TFLOPS


In [14]:
class BertLoRAForNLI(nn.Module):
    def __init__(self, model_name="roberta-base", num_labels=3):
        super().__init__()
        
        # Load base model
        self.base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name, 
            num_labels=num_labels
        )
        
        # LoRA Configuration
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            inference_mode=False,
            r=8,  # Rank
            lora_alpha=16,  # Scaling parameter
            lora_dropout=0.1,
            target_modules=["query", "value"],  # Apply LoRA to attention layers
        )
        
        # Apply LoRA to the model
        self.model = get_peft_model(self.base_model, lora_config)
        
        # Print trainable parameters
        self.model.print_trainable_parameters()

    def forward(self, input_ids, labels, attention_mask=None):
        """Forward pass compatible with your training loop
        
        RoBERTa doesn't use token_type_ids, so we ignore them
        """
        outputs = self.model(
            input_ids=input_ids,
            labels=labels,
            attention_mask=attention_mask
            # Note: token_type_ids removed for RoBERTa
        )
        return outputs.logits, outputs.loss
    
    def configure_optimizers(self, weight_decay, learning_rate, device, verbose=False):
        # Step 1: Collect trainable parameters (excluding frozen ones)
        param_dict = {pn: p for pn, p in self.named_parameters()}
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
    
        # Step 2: Separate weight decay (2D+ tensors) and no decay (biases, LayerNorm)
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]  # Weight matrices
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]  # Biases, LayerNorm
    
        # Step 3: Define optimizer parameter groups
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},  # Apply weight decay
            {'params': nodecay_params, 'weight_decay': 0.0}  # No weight decay
        ]

        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and 'cuda' in device
        
        # Debugging - Print parameter stats (optional)
        if verbose:
            num_decay_params = sum(p.numel() for p in decay_params)
            num_nodecay_params = sum(p.numel() for p in nodecay_params)
            print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params/1e6:.2f}M parameters")
            print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params/1e6:.2f}M parameters")
            print(f"Using fused AdamW: {use_fused}")

        # Create AdamW optimizer
        optimizer = torch.optim.AdamW(
            optim_groups, 
            lr=learning_rate, 
            betas=(0.9, 0.999),  # Momentum values
            eps=1e-8,  # Small epsilon for numerical stability
            fused=use_fused  # Enable fused version if available
        )
    
        return optimizer


# Initialize the model
model = BertLoRAForNLI(model_name="roberta-base", num_labels=3)
model = model.to(device)



def benchmark_model(model, tokenizer, text1, text2, seq_len=128, use_mixed_precision=True):
    """
    Benchmarks a BERT-based LoRA model (e.g., BertLoRAForNLI) using real tokenized inputs with a fixed sequence length.
    
    Args:
        model: LoRA-adapted BERT model (e.g., BertLoRAForNLI).
        tokenizer: Preloaded tokenizer (e.g., BertTokenizer).
        text1 (str): First input sentence (e.g., premise for NLI).
        text2 (str): Second input sentence (e.g., hypothesis for NLI).
        seq_len (int): Sequence length for tokenization (default: 128).
        use_mixed_precision (bool): Whether to use mixed precision (bf16/fp16) if on CUDA.
    
    Returns:
        List of throughput results for different batch sizes.
    """
    device = next(model.parameters()).device
    
    # Setup mixed precision
    if use_mixed_precision and device.type == "cuda":
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        print(f"Using mixed precision: {dtype}")
    else:
        dtype = torch.float32
        print("Using full precision: float32")
    
    # Prepare real tokenized inputs
    inputs = tokenizer(
        text1,
        text2,
        max_length=seq_len,  # Fixed to 128
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = inputs['input_ids'].to(device)  # Shape: [1, seq_len]
    # token_type_ids = inputs['token_type_ids'].to(device)  # Shape: [1, seq_len]
    attention_mask = inputs['attention_mask'].to(device)  # Shape: [1, seq_len]
    labels = torch.tensor([0], device=device)  # Example label (0 for contradiction)
    
    model.eval()
    
    print("="*70)
    print("MODEL STATISTICS")
    print("="*70)
    
    # Parameter Count
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
    
    # FLOPs Calculation
    print("\n" + "="*70)
    print("FLOPS ANALYSIS")
    print("="*70)
    
    with torch.no_grad():
        try:
            # For LoRA model, pass inputs matching the forward method
            flops = FlopCountAnalysis(model, (input_ids, labels, attention_mask))
            total_flops = flops.total()
        except Exception as e:
            print(f"FLOPs analysis with fvcore failed: {e}. Using approximate BERT FLOPs with LoRA adjustment.")
            # Approximate FLOPs for BERT-base with LoRA
            S, H, A, F, L = 128, 768, 12, 3072, 12  # seq_len, hidden_size, num_heads, ffn_size, num_layers
            base_flops = L * (4 * S * H**2 + 2 * S**2 * H + 4 * S * H * F)  # Attention + FFN
            # LoRA adds adapters to query and value: 2 * r * H per head per layer
            lora_params_per_layer = 2 * 8 * H * A  # r=8, query+value
            lora_flops = L * lora_params_per_layer * S  # Approximate FLOPs for LoRA adapters
            total_flops = base_flops + lora_flops
            print("Using approximate FLOPs calculation for LoRA model.")
        
        print(f"Total FLOPs per forward pass: {total_flops/1e9:.2f} GFLOPs")
        print(f"FLOPs per sample: {total_flops/1e9:.2f} GFLOPs")
        print(f"Total FLOPs for sequence: {total_flops/1e12:.4f} TFLOPs")
    
    # Memory Usage
    print("\n" + "="*70)
    print("MEMORY USAGE")
    print("="*70)
    
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        if use_mixed_precision and device.type == "cuda":
            with torch.autocast(device_type='cuda', dtype=dtype):
                _ = model(input_ids, labels, attention_mask)
        else:
            _ = model(input_ids, labels, attention_mask)
    
    if device.type == "cuda":
        memory_allocated = torch.cuda.max_memory_allocated() / 1e6
        print(f"Peak GPU memory (batch_size=1): {memory_allocated:.2f} MB")
    else:
        print("Memory usage not measured (running on CPU).")
    
    # Latency Measurement
    print("\n" + "="*70)
    print("LATENCY MEASUREMENT (batch_size=1)")
    print("="*70)
    
    # Warmup
    print("Warming up...")
    with torch.no_grad():
        for _ in range(20):
            if use_mixed_precision and device.type == "cuda":
                with torch.autocast(device_type='cuda', dtype=dtype):
                    _ = model(input_ids, labels, attention_mask)
            else:
                _ = model(input_ids, labels, attention_mask)
    if device.type == "cuda":
        torch.cuda.synchronize()
    
    # Actual measurement
    print("Measuring latency...")
    latencies = []
    num_iterations = 100
    
    with torch.no_grad():
        for _ in range(num_iterations):
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            
            if use_mixed_precision and device.type == "cuda":
                with torch.autocast(device_type='cuda', dtype=dtype):
                    _ = model(input_ids, labels, attention_mask)
            else:
                _ = model(input_ids, labels, attention_mask)
            
            if device.type == "cuda":
                torch.cuda.synchronize()
            latencies.append((time.perf_counter() - start) * 1000)  # ms
    
    latencies = np.array(latencies)
    mean_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    print(f"Latency: {mean_latency:.2f} ± {std_latency:.2f} ms")
    
    # Throughput Measurement
    print("\n" + "="*70)
    print("THROUGHPUT MEASUREMENT")
    print("="*70)
    
    batch_sizes = [1, 8, 16, 32]
    throughput_results = []
    
    for batch_size in batch_sizes:
        try:
            # Repeat tokenized inputs for larger batch sizes
            input_ids_batch = input_ids.repeat(batch_size, 1)
            attention_mask_batch = attention_mask.repeat(batch_size, 1)
            labels_batch = labels.repeat(batch_size)
            
            # Warmup
            with torch.no_grad():
                for _ in range(10):
                    if use_mixed_precision and device.type == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, labels_batch, attention_mask_batch)
                    else:
                        _ = model(input_ids_batch, labels_batch, attention_mask_batch)
            if device.type == "cuda":
                torch.cuda.synchronize()
            
            # Measure
            num_iterations = 50
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            
            with torch.no_grad():
                for _ in range(num_iterations):
                    if use_mixed_precision and device.type == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, labels_batch, attention_mask_batch)
                    else:
                        _ = model(input_ids_batch, labels_batch, attention_mask_batch)
            
            if device.type == "cuda":
                torch.cuda.synchronize()
            elapsed_time = time.perf_counter() - start
            
            samples_per_sec = (batch_size * num_iterations) / elapsed_time
            tokens_per_sec = samples_per_sec * seq_len  # Use fixed seq_len=128
            
            # Memory check
            if device.type == "cuda":
                torch.cuda.reset_peak_memory_stats()
                with torch.no_grad():
                    if use_mixed_precision and device.type == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, labels_batch, attention_mask_batch)
                    else:
                        _ = model(input_ids_batch, labels_batch, attention_mask_batch)
                memory_used = torch.cuda.max_memory_allocated() / 1e9
            else:
                memory_used = None
            
            throughput_results.append({
                'batch_size': batch_size,
                'samples_per_sec': samples_per_sec,
                'tokens_per_sec': tokens_per_sec,
                'memory_gb': memory_used
            })
            
            memory_str = f"{memory_used:.2f} GB" if memory_used else "N/A (CPU)"
            print(f"Batch size {batch_size:3d}: {samples_per_sec:7.2f} samples/sec | "
                  f"{tokens_per_sec:10.2f} tokens/sec | Memory: {memory_str}")
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"Batch size {batch_size:3d}: OOM (Out of Memory)")
                if device.type == "cuda":
                    torch.cuda.empty_cache()
                break
            else:
                raise e
    
    # Performance Summary
    print("\n" + "="*70)
    print("PERFORMANCE SUMMARY")
    print("="*70)
    
    if throughput_results:
        max_throughput = max(throughput_results, key=lambda x: x['tokens_per_sec'])
        print(f"Peak throughput: {max_throughput['tokens_per_sec']:.2f} tokens/sec "
              f"(batch_size={max_throughput['batch_size']})")
        print(f"Single sample latency: {mean_latency:.2f} ± {std_latency:.2f} ms")
        print(f"FLOPs per forward pass: {total_flops/1e9:.2f} GFLOPs")
        
        # Theoretical achieved FLOPS
        mean_latency_sec = mean_latency / 1000
        theoretical_flops = total_flops / mean_latency_sec / 1e12
        print(f"Achieved compute: {theoretical_flops:.2f} TFLOPS")
    
    return throughput_results



# Define input texts
text1 = "The man is walking down the street."
text2 = "A person is outside."

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

results = benchmark_model(model, tokenizer, text1, text2, seq_len=128, use_mixed_precision=True)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 887,811 || all params: 125,535,750 || trainable%: 0.7072


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Using mixed precision: torch.bfloat16
MODEL STATISTICS
Total parameters: 125,535,750 (125.54M)
Trainable parameters: 887,811 (0.89M)

FLOPS ANALYSIS
Total FLOPs per forward pass: 10.92 GFLOPs
FLOPs per sample: 10.92 GFLOPs
Total FLOPs for sequence: 0.0109 TFLOPs

MEMORY USAGE
Peak GPU memory (batch_size=1): 967.40 MB

LATENCY MEASUREMENT (batch_size=1)
Warming up...
Measuring latency...
Latency: 24.48 ± 1.22 ms

THROUGHPUT MEASUREMENT
Batch size   1:   43.22 samples/sec |    5532.26 tokens/sec | Memory: 0.97 GB
Batch size   8:  167.36 samples/sec |   21422.37 tokens/sec | Memory: 1.00 GB
Batch size  16:  186.97 samples/sec |   23931.99 tokens/sec | Memory: 1.04 GB
Batch size  32:  198.53 samples/sec |   25412.06 tokens/sec | Memory: 1.12 GB

PERFORMANCE SUMMARY
Peak throughput: 25412.06 tokens/sec (batch_size=32)
Single sample latency: 24.48 ± 1.22 ms
FLOPs per forward pass: 10.92 GFLOPs
Achieved compute: 0.45 TFLOPS
